In [1]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

load_dotenv(find_dotenv())

engine = create_engine(URL.create(
    "postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host="localhost",
    port=int(os.getenv("DB_PORT", "5432")),
    database=os.getenv("DB_NAME"),
))

df_raw = pd.read_sql("SELECT * FROM application_train", engine)
print(df_raw.shape)

(307511, 122)


In [2]:
df_raw

,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,flag_document_18,flag_document_19,flag_document_20,flag_document_21,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,0,0,0,0,1.0,0.0,0.0,1.0,0.0,1.0
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
print(df_raw.isna().sum().to_string())

sk_id_curr                           0
target                               0
name_contract_type                   0
code_gender                          0
flag_own_car                         0
flag_own_realty                      0
cnt_children                         0
amt_income_total                     0
amt_credit                           0
amt_annuity                         12
amt_goods_price                    278
name_type_suite                   1292
name_income_type                     0
name_education_type                  0
name_family_status                   0
name_housing_type                    0
region_population_relative           0
days_birth                           0
days_employed                        0
days_registration                    0
days_id_publish                      0
own_car_age                     202929
flag_mobil                           0
flag_emp_phone                       0
flag_work_phone                      0
flag_cont_mobile         

# Manual Encoding and Cleaning
  #### All categorical columns were inspected with `value_counts` and all numeric columns with a min/max/null summary, to catch unexpected categories, sentinel values and zero-variance fields. Individual outputs are omitted below; only columns requiring a decision are documented.  
* **name_contract_type** - Converted to Integer type.
* **code_gender** - Removed rare gender type(`XNA`) that only appears 4 times, Converted to Integer type.
* **flag_own_car** - Converted to Integer type.
* **flag_own_realty** - Converted to Integer type.
* **cnt_children** - capped at 5, Only 42 applicants exceed this, too few for a stable per-level default rate, and values up to 19 distort linear model coefficients.
Capping rather than dropping, since these are valid records and removing them would bias the sample.
* **amt_income_total** - Maximum of 117M against a median of 147,150. Rather than applying a threshold rule, I checked whether each extreme record was internally consistent. sk_id_curr = 114967 is not:a labourer with secondary education reporting 117M income, borrowing 562,491 at an annuity of 26,194, and then defaulting.
The credit-to-income ratio is 0.005 against a population median near 3, and the value is plausibly 117,000 recorded at 1000×.
Set to NaN rather than dropped, since the record's other 121 fields are valid.
The next four largest incomes are consistent with their occupations and credit amounts, so they were kept.

In [22]:
print(df['cnt_children'].value_counts())

cnt_children
0    215369
1     61118
2     26748
3      3717
4       429
5       126
Name: count, dtype: int64


In [40]:
odd = df_raw.nlargest(5, 'amt_income_total')
odd[['sk_id_curr', 'amt_income_total', 'amt_credit', 'amt_annuity',
     'name_income_type', 'occupation_type', 'organization_type',
     'name_education_type', 'target']]

,sk_id_curr,amt_income_total,amt_credit,amt_annuity,name_income_type,occupation_type,organization_type,name_education_type,target
12943,114967,117000000.0,562491.0,26194.5,Working,Laborers,Business Entity Type 3,Secondary / secondary special,1
203786,336147,18000090.0,675000.0,69295.5,Commercial associate,None,Business Entity Type 3,Secondary / secondary special,0
246955,385674,13500000.0,1400503.5,130945.5,Commercial associate,None,Business Entity Type 3,Higher education,0
77837,190160,9000000.0,1431531.0,132601.5,Working,Managers,Business Entity Type 1,Higher education,0
131199,252084,6750000.0,790830.0,52978.5,Working,Laborers,Transport: type 4,Higher education,0


In [42]:
import numpy as np
df = df_raw.copy()

name_contract_type_mapping = {'Cash loans' : 0, 'Revolving loans' : 1}
df['name_contract_type'] = df['name_contract_type'].map(name_contract_type_mapping)

df = df[~df['code_gender'].isin(['XNA'])].reset_index(drop=True)
code_gender_mapping = {'F' : 0, 'M' : 1}
df['code_gender'] = df['code_gender'].map(code_gender_mapping)

flag_own_car_mapping = {'N' : 0, 'Y' : 1}
df['flag_own_car'] = df['flag_own_car'].map(flag_own_car_mapping)

flag_own_realty_mapping = {'N' : 0, 'Y' : 1}
df['flag_own_realty'] = df['flag_own_realty'].map(flag_own_realty_mapping)

df['cnt_children'] = df['cnt_children'].clip(upper=5)

df.loc[df['sk_id_curr'] == 114967, 'amt_income_total'] = np.nan

In [21]:
df

,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,flag_document_18,flag_document_19,flag_document_20,flag_document_21,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year
0,100002,1,0,1,0,1,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,0,0,0,0,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,1,1,1,1,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,0,0,0,1,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,0,1,0,1,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307502,456251,0,0,1,0,0,0,157500.0,254700.0,27558.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307503,456252,0,0,0,0,1,0,72000.0,269550.0,12001.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307504,456253,0,0,0,0,1,0,153000.0,677664.0,29979.0,...,0,0,0,0,1.0,0.0,0.0,1.0,0.0,1.0
307505,456254,1,0,0,0,1,0,171000.0,370107.0,20205.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
